# Faker
---
[[site]](https://faker.readthedocs.io/en/master/)

Питоновская библиотека, предоставляющая набор удобных генераторов 

```pip install Faker```

In [2]:
from faker import Faker

fake = Faker()           # локаль по умолчанию (en_US)
fake_ru = Faker('ru_RU') # русская локаль


In [3]:
fake = Faker('ru_RU')

fake.name()           # 'Иван Кузнецов'
fake.first_name()     # 'Мария'
fake.last_name_male() # 'Смирнов'
fake.address()        # 'ул. Пушкина, д. 10, кв. 5, г. Москва...'
fake.email()          # 'pavel89@example.org'
fake.phone_number()   # '+7 (916) 123-45-67'
fake.company()        # 'ООО "АльфаТех"'
fake.job()            # 'Инженер-программист'
fake.date()           # '2022-11-03'
fake.date_between('-1y', 'today')  # дата за последний год
fake.url()            # 'https://example.net/strona'
fake.ipv4_private()   # '192.168.0.14'
fake.uuid4()          # '0b4b4d7c-...'


'e9c1dc98-fef6-4b28-98e7-72ac9e8e9d9d'

In [ ]:
from faker import Faker
Faker.seed(42)     # глобально
fake = Faker('ru_RU')
fake.seed_instance(42)  # локально для конкретного генератора


In [ ]:
fake = Faker('ru_RU')

fake.unique.email()   # каждый вызов — новое значение
# ...
fake.unique.clear()   # сбросить трекер уникальности (важно при больших объёмах)

fake.safe_email()     # домены вида example.com/org/net


In [ ]:
fake.name()
fake.first_name_female()
fake.last_name()
fake.date_of_birth(minimum_age=18, maximum_age=90)
fake.phone_number()
fake.email()
fake.user_name()
fake.password(length=12, special_chars=True)


In [ ]:
# география
fake.address()
fake.city()
fake.postcode()
fake.country()
fake.latitude(), fake.longitude()
fake.local_latlng(country_code="RU", coords_only=True)

# интернет-адреса
fake.url()
fake.uri()
fake.ipv4(), fake.ipv6()
fake.user_agent()
fake.mac_address()

# бизнес
fake.company()
fake.bs()                    # бизнес-слоган (en локаль)
fake.job()
fake.currency()              # ('RUB', 'Российский рубль') — зависит от локали
fake.iban()                  # фейковый IBAN
fake.credit_card_number()    # синтетический номер (небойся, тестовый)
fake.credit_card_full()

# интернет
fake.file_name(extension='csv')
fake.mime_type()
fake.color_name()      # 'LightSkyBlue'
fake.boolean(chance_of_getting_true=30)
fake.pyint(min_value=0, max_value=100)
fake.pyfloat(left_digits=2, right_digits=3, positive=True)
fake.pydecimal(left_digits=4, right_digits=2, positive=True)

# Локали
fake_ru = Faker('ru_RU')
fake_en = Faker('en_US')
fake_mix = Faker(['ru_RU', 'en_US'])
fake_mix.name()  # имя может быть русским или английским

# Типовые типы данных
fake.random_element(elements={'A': 0.7, 'B': 0.2, 'C': 0.1})
fake.random_int(min=1, max=10)
fake.pyfloat()  # см. выше
fake.date_between(start_date='-90d', end_date='-1d')



In [ ]:
from faker import Faker
fake = Faker('ru_RU')

def make_user():
    first = fake.first_name()
    last  = fake.last_name()
    return {
        'id': fake.uuid4(),
        'first_name': first,
        'last_name': last,
        'email': f"{first.lower()}.{last.lower()}@{fake.free_email_domain()}",
        'gender': fake.random_element(elements={'male':0.5, 'female':0.5}),
        'birthdate': fake.date_of_birth(18, 80),
        'city': fake.city(),
        'signup_ts': fake.date_time_between('-2y','now'),
        'is_active': fake.boolean(85),
    }

rows = [make_user() for _ in range(10_000)]


In [ ]:
import csv

with open('users.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)


In [ ]:
from faker.providers import BaseProvider

class CustomProvider(BaseProvider):
    def product_category(self):
        # распределение с перекосом к популярным категориям
        return self.random_element({
            'electronics': 0.35,
            'fashion':     0.25,
            'home':        0.20,
            'sports':      0.10,
            'other':       0.10,
        })

    def sku(self):
        return f"SKU-{self.random_int(100000, 999999)}"

fake = Faker('ru_RU')
fake.add_provider(CustomProvider)

fake.product_category()  # 'electronics'
fake.sku()               # 'SKU-834291'


In [ ]:
schema = {
    'order_id': lambda: fake.uuid4(),
    'user_id':  lambda: fake.uuid4(),
    'created_at': lambda: fake.date_time_between('-180d', 'now'),
    'status': lambda: fake.random_element({'new':0.5,'paid':0.3,'canceled':0.2}),
    'items_count': lambda: fake.random_int(1, 5),
    'total': lambda: round(fake.pyfloat(left_digits=3, right_digits=2, positive=True), 2),
}

def make_row(schema):
    return {k: gen() for k, gen in schema.items()}

orders = [make_row(schema) for _ in range(1000)]


users = []
for i in range(1000):
    uid = fake.uuid4()
    users.append({'user_id': uid, 'name': fake.name(), 'email': fake.safe_email()})

orders = []
for u in users:
    for _ in range(fake.random_int(0, 5)):
        orders.append({
            'order_id': fake.uuid4(),
            'user_id': u['user_id'],          # внешний ключ
            'created_at': fake.date_time_between('-90d', 'now'),
            'amount': round(fake.pyfloat(3, 2, True), 2),
        })


In [ ]:
from faker import Faker
fake = Faker()

# Можно из коробки генерировать типы данных
print("Random name=\n{}\n\nRandom address=\n{}\n\nRandom text=\n{}".format(
    fake.name(), 
    fake.address(), 
    fake.text())
)

fake.sentence()

my_word_list = [
'danish','cheesecake','sugar',
'Lollipop','wafer','Gummies',
'sesame','Jelly','beans',
'pie','bar','Ice','oat' ]
fake.sentence(ext_word_list=my_word_list)

fake.random
fake.random.getstate()

# метод unique переводит в режим генерации уникальных значений
names = [fake.unique.first_name() for i in range(500)]
assert len(set(names)) == len(names)

# если в этом режиме случается коллизия, выбрасываетсмя исключение
for i in range(3):
     # Raises a UniquenessException
     fake.unique.boolean()

locales = OrderedDict([
    ('en-US', 1),
    ('en-PH', 2),
    ('ja_JP', 3),
])
fake = Faker(locales)

for _ in range(10):
  print(fake.name())



### Упаржнение 1
сгенерировать 

In [ ]:
from faker import Faker
import pandas as pd, random
from datetime import datetime, timedelta

U, M, SEED = 50_000, 10, 42
fake = Faker("ru_RU")
random.seed(SEED); Faker.seed(SEED)

now, since = datetime.utcnow(), datetime.utcnow() - timedelta(days=730)
status = lambda r: "new" if r<0.5 else ("paid" if r<0.9 else "canceled")
amount = lambda: round(min(random.gammavariate(2,1500),200_000),2)

rows = []
for i in range(U):
    uid = fake.uuid4()
    fn, ln = fake.first_name(), fake.last_name()
    email = fake.unique.email()          # ✅ готовая уникальность
    city = fake.city()
    signed = fake.date_time_between(since, now)
    for _ in range(random.randint(0, M)):
        rows.append({
            "order_id": fake.uuid4(),
            "user_id": uid,
            "first_name": fn,
            "last_name": ln,
            "email": email,
            "city": city,
            "signup_ts": signed,
            "created_at": fake.date_time_between(signed, now),
            "status": status(random.random()),
            "amount": amount()
        })

df = pd.DataFrame(rows)
print(df.head(), "\nВсего заказов:", len(df))


In [ ]:
from faker import Faker
import random, pandas as pd
from datetime import datetime, timedelta

N, USERS, SEED = 100_000, 10_000, 123
fake = Faker(); random.seed(SEED); Faker.seed(SEED)

now = datetime.utcnow()
paths = ["/", "/catalog", "/product", "/cart", "/search", "/blog", "/about", "/contact"]
utm = {"direct":0.35,"seo":0.30,"ads":0.20,"email":0.10,"social":0.05}
devices = ["desktop","mobile","tablet"]
wchoice = lambda d: random.choices(list(d), weights=list(d.values()))[0]
skew = lambda: min(int(random.gammavariate(2.2, 15)), 3600)  # сек, среднее ~33

rows = []
for _ in range(N):
    ts = fake.date_time_between(start_date=now - timedelta(days=30), end_date=now)
    dur = skew()
    bounce = random.random() < (0.75 if dur < 10 else (0.25 if dur < 60 else 0.08))
    rows.append({
        "event_id": fake.uuid4(),
        "ts": ts,
        "session_id": fake.uuid4(),
        "user_id": random.randint(1, USERS),
        "page_path": random.choice(paths),
        "utm_source": wchoice(utm),
        "country": fake.country(),
        "ip": fake.ipv4_public(),
        "user_agent": fake.user_agent(),
        "device_type": random.choice(devices),
        "duration_sec": dur,
        "is_bounce": bounce,
    })

logs = pd.DataFrame(rows)
logs.head()
